# Module 4.1 — Modern Portfolio Theory
## The Geometry of Risk and Return

---

### On the Nature of Diversification

Modern Portfolio Theory begins with a deceptively simple observation: **risk and return cannot be understood in isolation**. An asset's risk is not an intrinsic property—it depends on what else you hold. A stock that is wildly volatile on its own may contribute almost nothing to a portfolio's risk if its movements are uncorrelated with the other holdings. This insight, formalized by Harry Markowitz in 1952, shattered the prevailing wisdom that rational investing meant simply picking the "best" individual securities. It replaced security selection with **portfolio construction**—a shift from one-dimensional thinking ("what is the best stock?") to the geometry of high-dimensional risk-return space.

The mathematical framework Markowitz developed is elegant in its foundations and profound in its implications. Given $N$ assets with expected returns $\boldsymbol{\mu} \in \mathbb{R}^N$ and covariance matrix $\boldsymbol{\Sigma} \in \mathbb{R}^{N \times N}$, the investor seeks weights $\mathbf{w} \in \mathbb{R}^N$ that trace out the **efficient frontier**—the set of portfolios achieving the maximum expected return for each level of risk, or equivalently, the minimum risk for each level of expected return. This is a quadratic optimization problem, and its solution reveals the fundamental trade-off that governs all of finance.

But the theory is not without its critics, and a mature quant must understand both its power and its fragility. Mean-variance optimization is exquisitely sensitive to its inputs: small errors in expected return estimates produce wildly different optimal portfolios. The covariance matrix must be estimated from historical data, which is backward-looking by nature. The assumption of normally distributed returns ignores fat tails—the very events that destroy portfolios. We will build the full Markowitz apparatus, extend it with the Capital Asset Pricing Model and the Black-Litterman framework, and at every step interrogate the assumptions that make the machinery work.

---

### Learning Objectives

By the end of this module, you will:

1. **Derive and implement** mean-variance optimization from first principles
2. **Construct and visualize** the efficient frontier, minimum variance portfolio, and tangency portfolio
3. **Compute and interpret** the Sharpe ratio, information ratio, and other risk-adjusted measures
4. **Understand** the Capital Market Line and the separation theorem
5. **Implement** the CAPM and interpret alpha, beta, and systematic vs. idiosyncratic risk
6. **Apply** the Black-Litterman model to blend market equilibrium with investor views
7. **Recognize** the practical limitations of mean-variance optimization and estimation error

---

### Module Contents

1. **The Mean-Variance Framework** — From utility theory to quadratic programming
2. **The Efficient Frontier** — The boundary of the possible
3. **Risk-Adjusted Performance** — Sharpe, Sortino, and the meaning of "alpha"
4. **CAPM and Factor Models** — Decomposing returns into systematic and idiosyncratic
5. **The Black-Litterman Model** — When equilibrium meets opinion
6. **Estimation Error and Robustness** — Why the optimal portfolio is rarely optimal

---

*"Diversification is the only free lunch in finance." — Harry Markowitz*

*But even free lunches have hidden costs. This module teaches you to find them.*

In [ ]:
# ========================================
# Initial Setup and Configuration
# ========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.optimize import minimize
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

PLOT_CONFIG = {
    'figure.figsize': (14, 7),
    'axes.titlesize': 16,
    'axes.labelsize': 13,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 11,
    'figure.dpi': 100,
    'lines.linewidth': 2,
}
plt.rcParams.update(PLOT_CONFIG)

COLORS = {
    'frontier': '#1565C0',
    'tangency': '#C62828',
    'mvp': '#2E7D32',
    'cml': '#F57C00',
    'assets': '#6A1B9A',
    'random': '#90CAF9',
    'bl': '#00838F',
    'neutral': '#455A64',
}

print("=" * 80)
print(" " * 16 + "MODULE 4.1: MODERN PORTFOLIO THEORY")
print("=" * 80)
print(f"✓ Environment configured successfully")
print(f"✓ Random seed: {RANDOM_SEED}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ Pandas version: {pd.__version__}")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 80)
print("\n📚 Ready to explore the geometry of risk and return!\n")

---

## 1. The Mean-Variance Framework

### 1.1 The Investor's Problem

Markowitz formulated portfolio selection as an optimization problem rooted in **expected utility theory**. An investor with a quadratic utility function (or, equivalently, an investor facing normally distributed returns) cares about exactly two quantities: the **expected return** and the **variance** of the portfolio. Higher expected return is desirable; higher variance is not. The investor's problem is:

$$\max_{\mathbf{w}} \quad \mathbf{w}^\top \boldsymbol{\mu} - \frac{\lambda}{2} \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$

subject to $\mathbf{w}^\top \mathbf{1} = 1$ (weights sum to one), where:

- $\mathbf{w} \in \mathbb{R}^N$ — portfolio weights
- $\boldsymbol{\mu} \in \mathbb{R}^N$ — vector of expected excess returns
- $\boldsymbol{\Sigma} \in \mathbb{R}^{N \times N}$ — covariance matrix of returns (symmetric, positive semi-definite)
- $\lambda > 0$ — risk aversion parameter

The portfolio's expected return and variance are:

$$\mu_p = \mathbf{w}^\top \boldsymbol{\mu}, \qquad \sigma_p^2 = \mathbf{w}^\top \boldsymbol{\Sigma} \mathbf{w}$$

### 1.2 The Minimum-Variance Portfolio

The **global minimum variance portfolio (MVP)** minimizes $\sigma_p^2$ without any constraint on expected return. Its analytical solution is:

$$\mathbf{w}_{\text{MVP}} = \frac{\boldsymbol{\Sigma}^{-1} \mathbf{1}}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}}$$

This portfolio lies at the leftmost point of the efficient frontier—the portfolio with the absolute minimum risk achievable through diversification alone. It requires no estimate of expected returns, only the covariance matrix. For this reason, many practitioners prefer it: one fewer source of estimation error.

### 1.3 The Tangency Portfolio and the Sharpe Ratio

When a risk-free asset with return $r_f$ is available, the investor can combine any risky portfolio with lending or borrowing at $r_f$. The set of attainable portfolios expands from the curved efficient frontier to a straight line in $(\sigma, \mu)$ space—the **Capital Market Line (CML)**. The optimal risky portfolio is the one that maximizes the slope of this line:

$$\text{Sharpe Ratio} = \frac{\mu_p - r_f}{\sigma_p}$$

This **tangency portfolio** has weights:

$$\mathbf{w}_{\text{tan}} = \frac{\boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}{\mathbf{1}^\top \boldsymbol{\Sigma}^{-1} (\boldsymbol{\mu} - r_f \mathbf{1})}$$

The **Two-Fund Separation Theorem** follows: every efficient portfolio is a combination of the tangency portfolio and the risk-free asset. The investor's risk aversion $\lambda$ determines only the *allocation* between these two—not the composition of the risky portfolio itself. This is a profound result: all investors, regardless of risk preferences, should hold the same risky portfolio.

### 1.4 The Efficient Frontier: Analytical Construction

The efficient frontier can be traced analytically using the **two-fund theorem for risky assets**: any portfolio on the frontier is a linear combination of any two distinct frontier portfolios. Defining the constants:

$$A = \mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \boldsymbol{\mu}, \quad B = \boldsymbol{\mu}^\top \boldsymbol{\Sigma}^{-1} \boldsymbol{\mu}, \quad C = \mathbf{1}^\top \boldsymbol{\Sigma}^{-1} \mathbf{1}, \quad D = BC - A^2$$

the minimum variance for a target return $\mu_p$ is:

$$\sigma_p^2 = \frac{C \mu_p^2 - 2A \mu_p + B}{D}$$

This is a **parabola** in $(\sigma_p^2, \mu_p)$ space, or a **hyperbola** in $(\sigma_p, \mu_p)$ space. The efficient frontier is the upper branch of this hyperbola.

In [ ]:
# ========================================
# Mean-Variance Optimization Engine
# ========================================

def generate_asset_universe(n_assets: int = 8, n_days: int = 756,
                             seed: int = 42) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Generate a realistic multi-asset universe with sector structure."""
    rng = np.random.RandomState(seed)

    names = ['US Equity', 'Intl Equity', 'EM Equity', 'US Bonds',
             'Corp Bonds', 'REITs', 'Commodities', 'Gold']
    ann_returns = np.array([0.10, 0.08, 0.12, 0.03, 0.05, 0.09, 0.06, 0.04])
    ann_vols = np.array([0.16, 0.18, 0.24, 0.04, 0.06, 0.20, 0.18, 0.15])

    # Correlation structure
    corr = np.array([
        [1.00, 0.75, 0.65, -0.10, 0.15, 0.60, 0.20, 0.00],
        [0.75, 1.00, 0.70, -0.05, 0.10, 0.55, 0.25, 0.05],
        [0.65, 0.70, 1.00, -0.15, 0.05, 0.50, 0.30, 0.10],
        [-0.10, -0.05, -0.15, 1.00, 0.80, 0.10, -0.05, 0.30],
        [0.15, 0.10, 0.05, 0.80, 1.00, 0.20, 0.00, 0.15],
        [0.60, 0.55, 0.50, 0.10, 0.20, 1.00, 0.25, 0.05],
        [0.20, 0.25, 0.30, -0.05, 0.00, 0.25, 1.00, 0.35],
        [0.00, 0.05, 0.10, 0.30, 0.15, 0.05, 0.35, 1.00],
    ])

    cov = np.outer(ann_vols, ann_vols) * corr
    daily_cov = cov / 252
    daily_mu = ann_returns / 252

    L = np.linalg.cholesky(daily_cov)
    Z = rng.normal(0, 1, (n_days, n_assets))
    daily_returns = daily_mu + Z @ L.T

    dates = pd.bdate_range(start='2022-01-03', periods=n_days)
    returns_df = pd.DataFrame(daily_returns, index=dates, columns=names)
    prices_df = (1 + returns_df).cumprod() * 100

    return returns_df, prices_df


class MeanVarianceOptimizer:
    """Complete mean-variance optimization engine implementing Markowitz theory."""

    def __init__(self, returns: pd.DataFrame, risk_free_rate: float = 0.04):
        self.returns = returns
        self.mu = returns.mean() * 252            # annualized expected returns
        self.cov = returns.cov() * 252             # annualized covariance
        self.rf = risk_free_rate
        self.n = len(self.mu)
        self.asset_names = list(returns.columns)
        self.cov_inv = np.linalg.inv(self.cov.values)

        # Frontier constants
        ones = np.ones(self.n)
        mu_vec = self.mu.values
        self.A = ones @ self.cov_inv @ mu_vec
        self.B = mu_vec @ self.cov_inv @ mu_vec
        self.C = ones @ self.cov_inv @ ones
        self.D = self.B * self.C - self.A**2

    def minimum_variance_portfolio(self) -> Dict:
        """Global minimum variance portfolio (analytical solution)."""
        ones = np.ones(self.n)
        w = self.cov_inv @ ones / (ones @ self.cov_inv @ ones)
        return self._portfolio_stats(w, 'Minimum Variance')

    def tangency_portfolio(self) -> Dict:
        """Maximum Sharpe ratio portfolio (analytical solution)."""
        excess = self.mu.values - self.rf
        ones = np.ones(self.n)
        w = self.cov_inv @ excess / (ones @ self.cov_inv @ excess)
        return self._portfolio_stats(w, 'Tangency (Max Sharpe)')

    def efficient_portfolio(self, target_return: float,
                            long_only: bool = False) -> Dict:
        """Portfolio on the efficient frontier achieving target return."""
        n = self.n
        if long_only:
            result = minimize(
                lambda w: w @ self.cov.values @ w,
                x0=np.ones(n) / n,
                method='SLSQP',
                bounds=[(0, 1)] * n,
                constraints=[
                    {'type': 'eq', 'fun': lambda w: np.sum(w) - 1},
                    {'type': 'eq', 'fun': lambda w: w @ self.mu.values - target_return},
                ],
            )
            w = result.x
        else:
            # Analytical solution via Lagrange multipliers
            mu_vec = self.mu.values
            ones = np.ones(n)
            lam1 = (self.C * target_return - self.A) / self.D
            lam2 = (self.B - self.A * target_return) / self.D
            w = self.cov_inv @ (lam1 * mu_vec + lam2 * ones)

        return self._portfolio_stats(w, f'Target μ={target_return:.1%}')

    def efficient_frontier(self, n_points: int = 100,
                            long_only: bool = False) -> pd.DataFrame:
        """Trace the efficient frontier."""
        mvp = self.minimum_variance_portfolio()
        mu_min = mvp['return']
        mu_max = self.mu.max() * 1.2
        target_returns = np.linspace(mu_min, mu_max, n_points)

        frontier = []
        for target in target_returns:
            try:
                port = self.efficient_portfolio(target, long_only=long_only)
                frontier.append({'return': port['return'], 'volatility': port['volatility'],
                                 'sharpe': port['sharpe']})
            except Exception:
                pass

        return pd.DataFrame(frontier)

    def random_portfolios(self, n_portfolios: int = 5000,
                          seed: int = 42) -> pd.DataFrame:
        """Generate random portfolios to visualize the feasible set."""
        rng = np.random.RandomState(seed)
        results = []
        for _ in range(n_portfolios):
            w = rng.dirichlet(np.ones(self.n))
            ret = w @ self.mu.values
            vol = np.sqrt(w @ self.cov.values @ w)
            sr = (ret - self.rf) / vol
            results.append({'return': ret, 'volatility': vol, 'sharpe': sr})
        return pd.DataFrame(results)

    def _portfolio_stats(self, weights: np.ndarray, name: str) -> Dict:
        ret = weights @ self.mu.values
        vol = np.sqrt(weights @ self.cov.values @ weights)
        sharpe = (ret - self.rf) / vol
        return {
            'name': name,
            'weights': pd.Series(weights, index=self.asset_names),
            'return': ret,
            'volatility': vol,
            'sharpe': sharpe,
        }


# --- Generate data and optimize ---

returns_df, prices_df = generate_asset_universe()
opt = MeanVarianceOptimizer(returns_df, risk_free_rate=0.04)

mvp = opt.minimum_variance_portfolio()
tangency = opt.tangency_portfolio()

print("=" * 80)
print("ASSET UNIVERSE")
print("=" * 80)
print(f"\n{'Asset':<16} {'Ann.Return':>10} {'Ann.Vol':>10} {'Sharpe':>8}")
print("-" * 48)
for name in opt.asset_names:
    ret = opt.mu[name]
    vol = np.sqrt(opt.cov.loc[name, name])
    sr = (ret - opt.rf) / vol
    print(f"{name:<16} {ret:>9.2%} {vol:>9.2%} {sr:>8.3f}")

print(f"\n{'=' * 80}")
print("KEY PORTFOLIOS")
print(f"{'=' * 80}")

for port in [mvp, tangency]:
    print(f"\n{port['name']}:")
    print(f"  Expected Return: {port['return']:.2%}")
    print(f"  Volatility:      {port['volatility']:.2%}")
    print(f"  Sharpe Ratio:    {port['sharpe']:.4f}")
    print(f"  Weights:")
    for asset, w in port['weights'].items():
        if abs(w) > 0.001:
            print(f"    {asset:<16} {w:>8.2%}")

# --- Efficient frontier visualization ---

frontier_uncon = opt.efficient_frontier(n_points=200, long_only=False)
frontier_con = opt.efficient_frontier(n_points=200, long_only=True)
random_ports = opt.random_portfolios(n_portfolios=8000)

fig, ax = plt.subplots(1, 1, figsize=(14, 9))

# Random portfolios (feasible set)
scatter = ax.scatter(random_ports['volatility'] * 100, random_ports['return'] * 100,
                     c=random_ports['sharpe'], cmap='viridis', alpha=0.3, s=5,
                     label='Random Portfolios')
plt.colorbar(scatter, ax=ax, label='Sharpe Ratio', shrink=0.7)

# Unconstrained frontier
ax.plot(frontier_uncon['volatility'] * 100, frontier_uncon['return'] * 100,
        color=COLORS['frontier'], linewidth=3, label='Efficient Frontier (Unconstrained)')

# Constrained frontier
ax.plot(frontier_con['volatility'] * 100, frontier_con['return'] * 100,
        color=COLORS['frontier'], linewidth=2, linestyle='--',
        label='Efficient Frontier (Long-Only)', alpha=0.8)

# Capital Market Line
cml_vol = np.linspace(0, frontier_uncon['volatility'].max() * 100, 100)
cml_ret = opt.rf * 100 + tangency['sharpe'] * cml_vol
ax.plot(cml_vol, cml_ret, color=COLORS['cml'], linewidth=2, linestyle='-.',
        label='Capital Market Line')

# Special portfolios
ax.scatter(mvp['volatility'] * 100, mvp['return'] * 100,
           color=COLORS['mvp'], s=200, marker='D', zorder=5, edgecolors='white',
           linewidth=2, label=f'MVP (σ={mvp["volatility"]:.1%})')
ax.scatter(tangency['volatility'] * 100, tangency['return'] * 100,
           color=COLORS['tangency'], s=200, marker='*', zorder=5, edgecolors='white',
           linewidth=2, label=f'Tangency (SR={tangency["sharpe"]:.3f})')

# Individual assets
for name in opt.asset_names:
    ret = opt.mu[name] * 100
    vol = np.sqrt(opt.cov.loc[name, name]) * 100
    ax.scatter(vol, ret, color=COLORS['assets'], s=80, zorder=4,
               edgecolors='white', linewidth=1)
    ax.annotate(name, (vol, ret), textcoords='offset points',
                xytext=(8, 4), fontsize=8, alpha=0.8)

# Risk-free rate
ax.scatter(0, opt.rf * 100, color='black', s=100, marker='s', zorder=5,
           label=f'Risk-Free Rate ({opt.rf:.0%})')

ax.set_title('The Geometry of Risk and Return: Efficient Frontier',
             fontsize=18, fontweight='bold')
ax.set_xlabel('Annualized Volatility (%)', fontsize=14)
ax.set_ylabel('Annualized Expected Return (%)', fontsize=14)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_xlim(-0.5, None)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## 2. The Capital Asset Pricing Model (CAPM)

### 2.1 From Portfolio Theory to Asset Pricing

If all investors hold the same tangency portfolio (as the separation theorem suggests), and markets clear, then the tangency portfolio must be the **market portfolio**—a value-weighted portfolio of all risky assets. This equilibrium argument yields the **Capital Asset Pricing Model** (Sharpe 1964, Lintner 1965):

$$\mathbb{E}[R_i] - r_f = \beta_i (\mathbb{E}[R_m] - r_f)$$

where:

$$\beta_i = \frac{\text{Cov}(R_i, R_m)}{\text{Var}(R_m)}$$

Beta measures an asset's **systematic risk**—the component of its risk that cannot be diversified away. An asset with $\beta = 1.5$ is expected to move 1.5% for every 1% move in the market. The CAPM says that in equilibrium, only systematic risk is compensated; idiosyncratic risk earns no premium because it can be diversified away.

### 2.2 Alpha: The Measure of Skill (or Luck)

In practice, we estimate the CAPM regression:

$$R_{i,t} - r_f = \alpha_i + \beta_i (R_{m,t} - r_f) + \varepsilon_{i,t}$$

The intercept $\alpha_i$ is **Jensen's alpha**—the return unexplained by market exposure. A positive alpha suggests the asset (or manager) outperforms after adjusting for systematic risk. Alpha is the holy grail of active management: it represents return from skill, information, or an unexploited anomaly. But beware: alpha estimated in-sample is often just noise. Out-of-sample, most alphas vanish.

### 2.3 Beyond Single-Factor: Multi-Factor Models

The CAPM's single market factor is insufficient to explain the cross-section of returns. Fama and French (1993) extended it to three factors:

$$R_{i,t} - r_f = \alpha_i + \beta_{i,m} (R_m - r_f) + \beta_{i,s} \text{SMB}_t + \beta_{i,v} \text{HML}_t + \varepsilon_{i,t}$$

where SMB (Small Minus Big) captures the size effect and HML (High Minus Low) captures the value effect. Carhart (1997) added a momentum factor (UMD), and Fama-French (2015) extended to five factors. The interpretation shifts: what looks like alpha in a single-factor model may be explained by exposure to other priced risk factors.

In [ ]:
# ========================================
# CAPM and Factor Analysis
# ========================================

class FactorModel:
    """CAPM and multi-factor regression model for return decomposition."""

    def __init__(self, returns: pd.DataFrame, market_col: str = 'US Equity',
                 rf_annual: float = 0.04):
        self.returns = returns
        self.rf_daily = rf_annual / 252
        self.market = returns[market_col] - self.rf_daily
        self.assets = [c for c in returns.columns if c != market_col]

    def capm(self) -> pd.DataFrame:
        """Estimate CAPM for all assets."""
        results = []
        for asset in self.assets:
            excess = self.returns[asset] - self.rf_daily
            slope, intercept, r_value, p_value, std_err = stats.linregress(
                self.market, excess
            )
            results.append({
                'Asset': asset,
                'Alpha (ann.)': intercept * 252,
                'Beta': slope,
                'R²': r_value**2,
                'Alpha p-val': p_value,
                'Tracking Error': (excess - slope * self.market - intercept).std() * np.sqrt(252),
                'Info Ratio': (intercept * 252) / ((excess - slope * self.market - intercept).std() * np.sqrt(252)),
                'Systematic Risk': slope**2 * self.market.var() * 252,
                'Idiosync. Risk': (excess - slope * self.market - intercept).var() * 252,
            })
        return pd.DataFrame(results).set_index('Asset')


# --- CAPM analysis ---

fm = FactorModel(returns_df)
capm_results = fm.capm()

print("=" * 80)
print("CAPM REGRESSION RESULTS")
print("=" * 80)
print(f"\nMarket proxy: US Equity | Risk-free: {opt.rf:.0%}/year")
print(f"\n{capm_results[['Alpha (ann.)', 'Beta', 'R²', 'Tracking Error', 'Info Ratio']].round(4).to_string()}")

# --- Visualization: CAPM & Security Market Line ---

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Security Market Line
ax = axes[0]
betas = capm_results['Beta'].values
returns_ann = [opt.mu[a] for a in capm_results.index]
market_ret = opt.mu['US Equity']
market_var = opt.cov.loc['US Equity', 'US Equity']

beta_range = np.linspace(-0.2, 1.8, 100)
sml = opt.rf + beta_range * (market_ret - opt.rf)
ax.plot(beta_range, sml * 100, color=COLORS['cml'], linewidth=2, label='Security Market Line')

for i, asset in enumerate(capm_results.index):
    color = COLORS['tangency'] if capm_results.loc[asset, 'Alpha (ann.)'] > 0 else COLORS['frontier']
    ax.scatter(betas[i], returns_ann[i] * 100, color=color, s=100, zorder=5,
               edgecolors='white', linewidth=1.5)
    ax.annotate(asset, (betas[i], returns_ann[i] * 100), textcoords='offset points',
                xytext=(8, 4), fontsize=9)

ax.scatter(1.0, market_ret * 100, color='black', s=150, marker='D', zorder=6,
           label='Market', edgecolors='white', linewidth=1.5)
ax.scatter(0, opt.rf * 100, color='black', s=100, marker='s', zorder=6, label='Risk-Free')

ax.set_title('Security Market Line (CAPM)', fontsize=16, fontweight='bold')
ax.set_xlabel('Beta (β)', fontsize=13)
ax.set_ylabel('Expected Return (%)', fontsize=13)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Risk decomposition
ax = axes[1]
assets = capm_results.index.tolist()
sys_risk = capm_results['Systematic Risk'].values * 100
idio_risk = capm_results['Idiosync. Risk'].values * 100
x_pos = np.arange(len(assets))

ax.bar(x_pos, sys_risk, color=COLORS['frontier'], label='Systematic Risk', alpha=0.8)
ax.bar(x_pos, idio_risk, bottom=sys_risk, color=COLORS['neutral'],
       label='Idiosyncratic Risk', alpha=0.8)
ax.set_xticks(x_pos)
ax.set_xticklabels(assets, rotation=45, ha='right', fontsize=9)
ax.set_title('Risk Decomposition: Systematic vs. Idiosyncratic', fontsize=16, fontweight='bold')
ax.set_ylabel('Variance (%²)', fontsize=13)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

---

## 3. The Black-Litterman Model

### 3.1 The Problem with Markowitz

Mean-variance optimization has a well-known pathology: it is an **error maximizer**. The optimizer aggressively exploits estimation errors in expected returns, producing extreme, concentrated, and unstable portfolios. A 1% change in one asset's expected return estimate can cause the optimal weights to flip sign entirely. As Michaud (1989) put it: "The problem with mean-variance optimization is not that it uses the wrong mathematics, but that it uses the wrong inputs."

The Black-Litterman model (1992) addresses this by replacing raw expected return estimates with a **Bayesian blend** of two sources of information:

1. **Market equilibrium returns** (the "prior"): the expected returns implied by the market portfolio's weights, assuming the market is mean-variance efficient
2. **Investor views** (the "likelihood"): the investor's opinions about specific asset or relative returns, expressed with varying degrees of confidence

### 3.2 The Mathematical Framework

**Step 1 — Implied equilibrium returns** (reverse optimization):

$$\boldsymbol{\Pi} = \lambda \boldsymbol{\Sigma} \mathbf{w}_{\text{mkt}}$$

where $\lambda = \frac{\mu_m - r_f}{\sigma_m^2}$ is the market risk aversion and $\mathbf{w}_{\text{mkt}}$ are market capitalization weights.

**Step 2 — Express views** as a system: $\mathbf{P} \boldsymbol{\mu} = \mathbf{q} + \boldsymbol{\varepsilon}$, where $\mathbf{P}$ is the pick matrix, $\mathbf{q}$ is the vector of view returns, and $\boldsymbol{\varepsilon} \sim N(\mathbf{0}, \boldsymbol{\Omega})$ represents view uncertainty.

**Step 3 — Posterior (blended) returns**:

$$\hat{\boldsymbol{\mu}} = \left[(\tau \boldsymbol{\Sigma})^{-1} + \mathbf{P}^\top \boldsymbol{\Omega}^{-1} \mathbf{P}\right]^{-1} \left[(\tau \boldsymbol{\Sigma})^{-1} \boldsymbol{\Pi} + \mathbf{P}^\top \boldsymbol{\Omega}^{-1} \mathbf{q}\right]$$

where $\tau$ is a scalar controlling the weight given to the prior (typically 0.025–0.05).

The beauty of Black-Litterman is its **graceful degradation**: with no views ($\mathbf{P}$ empty), it returns market equilibrium weights. As views are added with high confidence, the portfolio tilts toward those views. Uncertain views barely move the needle. This produces portfolios that are **stable, diversified, and intuitive**—a dramatic improvement over raw Markowitz.

In [ ]:
# ========================================
# Black-Litterman Model
# ========================================

class BlackLitterman:
    """Black-Litterman model for combining market equilibrium with investor views."""

    def __init__(self, cov: np.ndarray, market_weights: np.ndarray,
                 risk_free: float = 0.04, market_return: float = 0.10,
                 tau: float = 0.05, asset_names: List[str] = None):
        self.cov = cov
        self.w_mkt = market_weights
        self.rf = risk_free
        self.tau = tau
        self.asset_names = asset_names or [f'Asset_{i}' for i in range(len(market_weights))]

        # Market risk aversion
        mkt_var = market_weights @ cov @ market_weights
        self.lam = (market_return - risk_free) / mkt_var

        # Implied equilibrium returns
        self.pi = self.lam * cov @ market_weights

    def posterior(self, P: np.ndarray, Q: np.ndarray,
                 omega: np.ndarray = None) -> Tuple[np.ndarray, np.ndarray]:
        """
        Compute posterior expected returns given views.

        Parameters
        ----------
        P : (K, N) pick matrix — which assets are involved in each view
        Q : (K,) view returns
        omega : (K, K) view uncertainty. If None, uses tau * P @ Sigma @ P'
        """
        if omega is None:
            omega = np.diag(np.diag(self.tau * P @ self.cov @ P.T))

        tau_sigma = self.tau * self.cov
        tau_sigma_inv = np.linalg.inv(tau_sigma)
        omega_inv = np.linalg.inv(omega)

        # Posterior precision and mean
        posterior_precision = tau_sigma_inv + P.T @ omega_inv @ P
        posterior_cov = np.linalg.inv(posterior_precision)
        posterior_mu = posterior_cov @ (tau_sigma_inv @ self.pi + P.T @ omega_inv @ Q)

        return posterior_mu, posterior_cov

    def optimal_weights(self, posterior_mu: np.ndarray) -> np.ndarray:
        """Compute optimal weights from posterior returns."""
        w = np.linalg.inv(self.lam * self.cov) @ posterior_mu
        w = w / w.sum()  # normalize
        return w


# --- Set up Black-Litterman ---

# Market cap weights (simulated)
mkt_caps = np.array([30, 20, 10, 15, 8, 5, 7, 5], dtype=float)
w_mkt = mkt_caps / mkt_caps.sum()

bl = BlackLitterman(
    cov=opt.cov.values,
    market_weights=w_mkt,
    risk_free=0.04,
    market_return=0.10,
    tau=0.05,
    asset_names=opt.asset_names,
)

print("=" * 80)
print("BLACK-LITTERMAN MODEL")
print("=" * 80)

print(f"\nMarket risk aversion (λ): {bl.lam:.4f}")
print(f"Uncertainty scaling (τ): {bl.tau}")

print(f"\n{'Asset':<16} {'Mkt Weight':>10} {'Implied μ':>10} {'Sample μ':>10} {'Diff':>8}")
print("-" * 58)
for i, name in enumerate(opt.asset_names):
    print(f"{name:<16} {w_mkt[i]:>9.1%} {bl.pi[i]:>9.2%} {opt.mu.values[i]:>9.2%} "
          f"{(opt.mu.values[i] - bl.pi[i]):>7.2%}")

# --- Define investor views ---

# View 1: EM Equity will outperform Intl Equity by 3% (relative view)
# View 2: Gold will return 6% (absolute view)
# View 3: US Equity will outperform REITs by 2% (relative view)

n = len(opt.asset_names)
P = np.zeros((3, n))
Q = np.zeros(3)

# View 1: EM Equity - Intl Equity = 3%
P[0, opt.asset_names.index('EM Equity')] = 1
P[0, opt.asset_names.index('Intl Equity')] = -1
Q[0] = 0.03

# View 2: Gold = 6%
P[1, opt.asset_names.index('Gold')] = 1
Q[1] = 0.06

# View 3: US Equity - REITs = 2%
P[2, opt.asset_names.index('US Equity')] = 1
P[2, opt.asset_names.index('REITs')] = -1
Q[2] = 0.02

print("\nInvestor Views:")
print("  1. EM Equity will outperform Intl Equity by 3%")
print("  2. Gold will return 6%")
print("  3. US Equity will outperform REITs by 2%")

# Compute posterior
posterior_mu, posterior_cov = bl.posterior(P, Q)
bl_weights = bl.optimal_weights(posterior_mu)

print(f"\n{'=' * 80}")
print("POSTERIOR RETURNS AND WEIGHTS")
print(f"{'=' * 80}")
print(f"\n{'Asset':<16} {'Prior μ':>10} {'Posterior μ':>12} {'Shift':>8} {'Mkt Wt':>8} {'BL Wt':>8}")
print("-" * 66)
for i, name in enumerate(opt.asset_names):
    print(f"{name:<16} {bl.pi[i]:>9.2%} {posterior_mu[i]:>11.2%} "
          f"{(posterior_mu[i] - bl.pi[i]):>7.2%} {w_mkt[i]:>7.1%} {bl_weights[i]:>7.1%}")

# --- Visualization ---

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Expected returns comparison
ax = axes[0]
x = np.arange(n)
width = 0.25
ax.barh(x - width, bl.pi * 100, width, color=COLORS['neutral'], label='Equilibrium (Prior)', alpha=0.8)
ax.barh(x, posterior_mu * 100, width, color=COLORS['bl'], label='Black-Litterman (Posterior)', alpha=0.8)
ax.barh(x + width, opt.mu.values * 100, width, color=COLORS['tangency'], label='Sample (Historical)', alpha=0.8)
ax.set_yticks(x)
ax.set_yticklabels(opt.asset_names, fontsize=10)
ax.set_xlabel('Expected Return (%)', fontsize=13)
ax.set_title('Expected Returns: Three Perspectives', fontsize=16, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

# Weights comparison
ax = axes[1]
ax.barh(x - width, w_mkt * 100, width, color=COLORS['neutral'], label='Market Cap', alpha=0.8)
ax.barh(x, bl_weights * 100, width, color=COLORS['bl'], label='Black-Litterman', alpha=0.8)

# Markowitz tangency for comparison
tang_w = tangency['weights'].values
ax.barh(x + width, tang_w * 100, width, color=COLORS['tangency'], label='Markowitz Tangency', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(opt.asset_names, fontsize=10)
ax.set_xlabel('Weight (%)', fontsize=13)
ax.set_title('Portfolio Weights: BL vs. Market vs. Markowitz', fontsize=16, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\n💡 Black-Litterman weights are stable and diversified — a dramatic")
print("   contrast to the extreme positions of unconstrained Markowitz.")
print("   Views tilt the portfolio gently; the equilibrium anchor prevents overreaction.")

---

## 4. Estimation Error, Robustness, and Practical Considerations

### 4.1 The Fragility of Optimized Portfolios

The greatest weakness of mean-variance optimization is its sensitivity to **estimation error** in the inputs—particularly expected returns. The covariance matrix, estimated from historical data with hundreds of observations, is relatively stable. But expected returns, which drive portfolio selection far more than covariances, are notoriously difficult to estimate. A 1% error in one asset's expected return can flip the sign of its optimal weight.

This fragility manifests in several ways:

1. **Weight instability**: Small changes in inputs produce large changes in optimal weights
2. **Concentration**: The optimizer loads heavily into assets with (possibly erroneous) high return estimates
3. **Poor out-of-sample performance**: In-sample optimal portfolios often underperform naive strategies (like equal-weight) out of sample
4. **Extreme leverage**: Unconstrained solutions may require massive short positions

### 4.2 Practical Defenses

The field has developed several defenses against estimation error:

- **Constraints**: Long-only, maximum position size, sector limits. These regularize the problem.
- **Shrinkage estimators**: The Ledoit-Wolf shrinkage covariance estimator blends the sample covariance with a structured target, reducing estimation noise.
- **Resampled efficiency** (Michaud): Simulate many return scenarios from the estimated distribution, optimize each, and average the resulting portfolios.
- **Bayesian approaches**: The Black-Litterman model is itself a shrinkage method—it pulls returns toward the equilibrium prior.
- **Equal-weight and 1/N**: When estimation error dominates, the naive 1/N portfolio is surprisingly hard to beat.

### 4.3 Rebalancing: Theory Meets Transaction Costs

The efficient frontier is a static concept—it describes the optimal portfolio *at a single point in time*. In reality, portfolios drift as asset prices change, and the optimal weights shift as new data arrives. **Rebalancing** restores the portfolio to its target allocation, but each trade incurs transaction costs (commissions, bid-ask spread, market impact). The optimal rebalancing strategy balances the cost of deviating from the target against the cost of trading.

Common approaches:
- **Calendar rebalancing**: Rebalance at fixed intervals (monthly, quarterly)
- **Threshold rebalancing**: Rebalance when any weight drifts beyond a tolerance band
- **Optimal control**: Solve for the rebalancing policy that maximizes net-of-cost utility

In [ ]:
# ========================================
# Estimation Error & Portfolio Comparison
# ========================================

def bootstrap_frontier_stability(returns: pd.DataFrame, n_bootstrap: int = 200,
                                  rf: float = 0.04, seed: int = 42) -> Dict:
    """Demonstrate how the tangency portfolio changes with small perturbations."""
    rng = np.random.RandomState(seed)
    n_days, n_assets = returns.shape
    tangency_weights = []
    tangency_returns = []
    tangency_vols = []

    for _ in range(n_bootstrap):
        idx = rng.choice(n_days, size=n_days, replace=True)
        boot_returns = returns.iloc[idx]
        boot_opt = MeanVarianceOptimizer(boot_returns, risk_free_rate=rf)
        try:
            t = boot_opt.tangency_portfolio()
            tangency_weights.append(t['weights'].values)
            tangency_returns.append(t['return'])
            tangency_vols.append(t['volatility'])
        except Exception:
            pass

    return {
        'weights': np.array(tangency_weights),
        'returns': np.array(tangency_returns),
        'vols': np.array(tangency_vols),
    }


def backtest_strategies(returns: pd.DataFrame, rf: float = 0.04,
                         rebalance_freq: int = 21, lookback: int = 252) -> pd.DataFrame:
    """Backtest several portfolio strategies with monthly rebalancing."""
    n_days = len(returns)
    n_assets = returns.shape[1]
    strategies = {
        'Equal Weight': [],
        'Min Variance': [],
        'Max Sharpe': [],
        'Inv. Volatility': [],
    }

    current_weights = {s: np.ones(n_assets) / n_assets for s in strategies}

    for t in range(lookback, n_days):
        if (t - lookback) % rebalance_freq == 0:
            hist = returns.iloc[t - lookback:t]
            try:
                o = MeanVarianceOptimizer(hist, risk_free_rate=rf)

                # Equal weight
                current_weights['Equal Weight'] = np.ones(n_assets) / n_assets

                # Min variance (long-only)
                mvp_p = o.efficient_portfolio(o.minimum_variance_portfolio()['return'], long_only=True)
                current_weights['Min Variance'] = mvp_p['weights'].values

                # Max Sharpe (long-only)
                frontier = o.efficient_frontier(n_points=50, long_only=True)
                best_idx = frontier['sharpe'].idxmax()
                target_ret = frontier.loc[best_idx, 'return']
                max_sr = o.efficient_portfolio(target_ret, long_only=True)
                current_weights['Max Sharpe'] = max_sr['weights'].values

                # Inverse volatility
                vols = hist.std().values * np.sqrt(252)
                inv_vol = 1 / vols
                current_weights['Inv. Volatility'] = inv_vol / inv_vol.sum()

            except Exception:
                pass

        # Compute daily return for each strategy
        daily_ret = returns.iloc[t].values
        for s in strategies:
            strategies[s].append(current_weights[s] @ daily_ret)

    dates = returns.index[lookback:]
    return pd.DataFrame(strategies, index=dates)


# --- Bootstrap stability analysis ---

print("=" * 80)
print("ESTIMATION ERROR: TANGENCY PORTFOLIO INSTABILITY")
print("=" * 80)

boot = bootstrap_frontier_stability(returns_df, n_bootstrap=300)

print(f"\nBootstrap samples: {len(boot['returns'])}")
print(f"\n{'Asset':<16} {'Mean Wt':>10} {'Std Wt':>10} {'Min':>10} {'Max':>10}")
print("-" * 60)
for i, name in enumerate(opt.asset_names):
    ws = boot['weights'][:, i]
    print(f"{name:<16} {ws.mean():>9.1%} {ws.std():>9.1%} {ws.min():>9.1%} {ws.max():>9.1%}")

print(f"\n💡 The tangency portfolio weights swing wildly across bootstrap samples.")
print(f"   This is the estimation error problem: small data changes → big weight changes.")

# --- Backtest comparison ---

print(f"\n{'=' * 80}")
print("OUT-OF-SAMPLE STRATEGY COMPARISON")
print(f"{'=' * 80}")

bt_results = backtest_strategies(returns_df)

print(f"\n{'Strategy':<18} {'Ann.Ret':>10} {'Ann.Vol':>10} {'Sharpe':>8} {'MaxDD':>8}")
print("-" * 58)

for col in bt_results.columns:
    rets = bt_results[col]
    ann_ret = rets.mean() * 252
    ann_vol = rets.std() * np.sqrt(252)
    sharpe = (ann_ret - opt.rf) / ann_vol if ann_vol > 0 else 0
    cum = (1 + rets).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    print(f"{col:<18} {ann_ret:>9.2%} {ann_vol:>9.2%} {sharpe:>8.3f} {max_dd:>7.2%}")

# --- Visualization ---

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle("Estimation Error and Strategy Robustness",
             fontsize=18, fontweight='bold', y=1.02)

# Panel 1: Bootstrap tangency weight distributions
ax = axes[0, 0]
bp = ax.boxplot([boot['weights'][:, i] * 100 for i in range(len(opt.asset_names))],
                labels=opt.asset_names, patch_artist=True, vert=True)
colors_box = plt.cm.viridis(np.linspace(0.2, 0.8, len(opt.asset_names)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set_xticklabels(opt.asset_names, rotation=45, ha='right', fontsize=8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Tangency Portfolio Weight Instability (300 Bootstraps)', fontsize=13)
ax.set_ylabel('Weight (%)')

# Panel 2: Bootstrap tangency on risk-return plane
ax = axes[0, 1]
ax.scatter(boot['vols'] * 100, boot['returns'] * 100, alpha=0.3, s=15,
           color=COLORS['random'])
ax.scatter(tangency['volatility'] * 100, tangency['return'] * 100,
           color=COLORS['tangency'], s=200, marker='*', zorder=5, edgecolors='white',
           linewidth=2, label='Full-Sample Tangency')
ax.set_title('Tangency Portfolio Across Bootstrap Samples', fontsize=13)
ax.set_xlabel('Volatility (%)')
ax.set_ylabel('Return (%)')
ax.legend(fontsize=9)

# Panel 3: Cumulative returns
ax = axes[1, 0]
cum_returns = (1 + bt_results).cumprod()
for col in cum_returns.columns:
    ax.plot(cum_returns.index, cum_returns[col], linewidth=1.5, label=col)
ax.set_title('Cumulative Returns: Out-of-Sample', fontsize=13)
ax.set_ylabel('Growth of $1')
ax.legend(fontsize=9, loc='upper left')

# Panel 4: Rolling Sharpe comparison
ax = axes[1, 1]
roll_window = 126
for col in bt_results.columns:
    rolling_sr = (bt_results[col].rolling(roll_window).mean() * 252 - opt.rf) / \
                 (bt_results[col].rolling(roll_window).std() * np.sqrt(252))
    ax.plot(rolling_sr.index, rolling_sr, linewidth=1.2, label=col, alpha=0.8)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title(f'{roll_window}-Day Rolling Sharpe Ratio', fontsize=13)
ax.set_ylabel('Sharpe Ratio')
ax.legend(fontsize=9, loc='upper left')

plt.tight_layout()
plt.show()


# ========================================
# Module Summary
# ========================================

print("\n" + "=" * 80)
print(" " * 18 + "MODULE 4.1 — SUMMARY")
print("=" * 80)
print("""
Key Concepts Covered:

  1. MEAN-VARIANCE OPTIMIZATION
     • The investor's problem: maximize return for a given level of risk
     • MVP requires only the covariance matrix; tangency also needs μ
     • The efficient frontier is the upper branch of a hyperbola in (σ, μ) space
     • Two-Fund Separation: all investors hold the same risky portfolio

  2. CAPM AND FACTOR MODELS
     • Beta measures systematic (non-diversifiable) risk
     • Alpha is the return unexplained by factor exposure
     • Multi-factor models (Fama-French) decompose returns more completely
     • Idiosyncratic risk earns no premium in equilibrium

  3. BLACK-LITTERMAN
     • Blends market equilibrium (prior) with investor views (likelihood)
     • Produces stable, diversified portfolios that tilt toward views
     • Graceful degradation: no views → market portfolio
     • Solves the error-maximization problem of raw Markowitz

  4. ESTIMATION ERROR AND ROBUSTNESS
     • Markowitz is an error maximizer: garbage in, garbage out
     • Bootstrap analysis reveals terrifying weight instability
     • Simple strategies (equal-weight, inverse-vol) are hard to beat OOS
     • Constraints, shrinkage, and Bayesian methods are practical defenses

  The deepest lesson of portfolio theory is not how to find the optimal
  portfolio—it is understanding why the "optimal" portfolio is almost
  never optimal in practice, and what to do about it.
""")
print("=" * 80)
print("\n📚 Next: Module 4.2 — Risk Metrics & Management\n")